# QuantAlpha AI — Step 3C: Fundamental Quality Backtest (Long-Term / Position Mode)

Pure technical timing showed no real edge on Nifty 200 (Step 3B result: all signals under 2
points, inconsistent). This is expected — technical signals on a liquid, well-covered universe
are heavily arbitraged.

**This notebook tests a different, better-supported thesis: quality investing.** Academic and
practitioner research (Piotroski's original 2000 paper, Novy-Marx, and others) has consistently
found that high-quality companies (strong F-Score, high ROCE, healthy cash flow) tend to
outperform over **medium-to-long holding periods (3-12 months)** — a much stronger empirical
basis than short-term technical timing.

**What we test:**
1. Do stocks with high Piotroski F-Score (7-9) outperform low F-Score stocks (0-3) over 3/6/12 months?
2. Do stocks with high ROCE outperform low ROCE stocks over the same periods?
3. Combined: does (high F-Score AND high ROCE) show a stronger edge than either alone?

**Important honesty note:** this is still a SIMPLIFIED backtest. It uses TODAY's fundamental
scores applied against historical price movements — not true point-in-time fundamentals (i.e.,
what the F-Score actually WAS 6-12 months ago). This means there's look-ahead bias: today's score
reflects the company NOW, but we're checking it against past price action. Read the result as a
"does quality correlate with strong stocks generally" sanity check, not a fully rigorous backtest.
A fully rigorous version needs historical quarterly fundamental snapshots — a bigger future step,
noted at the end.


## 1. Connect to Drive + load data

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')

import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

fundamental_scores = pd.read_csv('data/fundamental_scores.csv')
fundamental_scores['symbol_short'] = fundamental_scores['symbol'].str.replace('.NS', '', regex=False)
print(f"Loaded fundamental scores for {len(fundamental_scores)} stocks")
fundamental_scores[['symbol_short', 'piotroski_f_score', 'roce']].describe()


Mounted at /content/drive
Loaded fundamental scores for 200 stocks


,piotroski_f_score,roce
count,200.000000,164.000000
mean,6.645000,0.233884
std,1.410291,0.390507
min,2.000000,-0.117589
25%,6.000000,0.141680
50%,7.000000,0.186120
75%,8.000000,0.246005
max,9.000000,4.971763


## 2. Compute forward returns for each stock
For each stock, calculate the actual price return over the last 3, 6, and 12 months of available
history — this is what we'll check against F-Score and ROCE.


In [2]:
def compute_forward_returns(symbol):
    path = f"data/technical/{symbol}.parquet"
    if not os.path.exists(path):
        return None
    df = pd.read_parquet(path)
    if len(df) < 260:  # need at least ~1 year of data
        return None
    df = df.reset_index(drop=True)

    current_price = df['Close'].iloc[-1]

    def price_n_days_ago(n):
        idx = len(df) - 1 - n
        return df['Close'].iloc[idx] if idx >= 0 else np.nan

    price_3m_ago = price_n_days_ago(63)   # ~3 months of trading days
    price_6m_ago = price_n_days_ago(126)  # ~6 months
    price_12m_ago = price_n_days_ago(252) # ~12 months

    return {
        'symbol_short': symbol,
        'return_3m': (current_price - price_3m_ago) / price_3m_ago if pd.notna(price_3m_ago) else np.nan,
        'return_6m': (current_price - price_6m_ago) / price_6m_ago if pd.notna(price_6m_ago) else np.nan,
        'return_12m': (current_price - price_12m_ago) / price_12m_ago if pd.notna(price_12m_ago) else np.nan,
    }

technical_files = glob.glob('data/technical/*.parquet')
symbols = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_files)

returns_data = []
for sym in symbols:
    r = compute_forward_returns(sym)
    if r:
        returns_data.append(r)

returns_df = pd.DataFrame(returns_data)
print(f"Computed historical returns for {len(returns_df)} stocks")
returns_df.describe()


Computed historical returns for 194 stocks


,return_3m,return_6m,return_12m
count,194.000000,194.000000,194.000000
mean,0.128771,0.020067,0.060013
std,0.178416,0.201769,0.288320
min,-0.593623,-0.528321,-0.552972
25%,0.012820,-0.108596,-0.122033
50%,0.113756,0.011313,0.016291
75%,0.234923,0.120606,0.212357
max,0.689502,0.695248,1.131626


## 3. Merge fundamental scores with returns

In [3]:
merged = fundamental_scores.merge(returns_df, on='symbol_short', how='inner')
print(f"Merged dataset: {len(merged)} stocks with both fundamentals and returns")
merged[['symbol_short', 'piotroski_f_score', 'roce', 'return_6m', 'return_12m']].head()


Merged dataset: 194 stocks with both fundamentals and returns


,symbol_short,piotroski_f_score,roce,return_6m,return_12m
0,360ONE,3,0.171741,-0.063494,-0.061958
1,ABB,5,0.176629,0.349889,0.149950
2,ABCAPITAL,4,NaN,0.121750,0.450651
3,ADANIENSOL,7,0.198309,0.527036,0.779189
4,ADANIENT,4,0.113337,0.434807,0.226822


## 4. Test 1 — Piotroski F-Score buckets vs returns
Splits stocks into F-Score groups (Low: 0-3, Mid: 4-6, High: 7-9) and compares average returns.


In [4]:
def fscore_bucket(score):
    if pd.isna(score):
        return 'unknown'
    if score <= 3:
        return 'Low (0-3)'
    elif score <= 6:
        return 'Mid (4-6)'
    else:
        return 'High (7-9)'

merged['fscore_bucket'] = merged['piotroski_f_score'].apply(fscore_bucket)

print("=== Returns by Piotroski F-Score bucket ===\n")
for period in ['return_3m', 'return_6m', 'return_12m']:
    print(f"--- {period} ---")
    summary = merged.groupby('fscore_bucket')[period].agg(['mean', 'median', 'count'])
    summary['mean'] = (summary['mean'] * 100).round(2)
    summary['median'] = (summary['median'] * 100).round(2)
    summary = summary.reindex(['Low (0-3)', 'Mid (4-6)', 'High (7-9)'])
    print(summary.to_string())
    print()


=== Returns by Piotroski F-Score bucket ===

--- return_3m ---
                mean  median  count
fscore_bucket                      
Low (0-3)      20.06   20.06      2
Mid (4-6)      12.28   11.17     81
High (7-9)     13.19   11.72    111

--- return_6m ---
               mean  median  count
fscore_bucket                     
Low (0-3)      3.80    3.80      2
Mid (4-6)     -0.47   -0.43     81
High (7-9)     3.78    4.59    111

--- return_12m ---
               mean  median  count
fscore_bucket                     
Low (0-3)      6.69    6.69      2
Mid (4-6)      2.63   -1.63     81
High (7-9)     8.45    2.48    111



## 5. Test 2 — ROCE quartiles vs returns
Splits stocks into ROCE quartiles (bottom 25%, middle 50%, top 25%) and compares returns.


In [5]:
merged['roce_quartile'] = pd.qcut(merged['roce'], q=4, labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'], duplicates='drop')

print("=== Returns by ROCE quartile ===\n")
for period in ['return_3m', 'return_6m', 'return_12m']:
    print(f"--- {period} ---")
    summary = merged.groupby('roce_quartile', observed=True)[period].agg(['mean', 'median', 'count'])
    summary['mean'] = (summary['mean'] * 100).round(2)
    summary['median'] = (summary['median'] * 100).round(2)
    print(summary.to_string())
    print()


=== Returns by ROCE quartile ===

--- return_3m ---
                mean  median  count
roce_quartile                      
Q1 (lowest)    14.53   13.11     40
Q2             13.57   12.96     39
Q3             13.07   10.57     39
Q4 (highest)   11.39   11.07     40

--- return_6m ---
               mean  median  count
roce_quartile                     
Q1 (lowest)   -0.17   -0.42     40
Q2             4.84    5.61     39
Q3             3.70   -0.29     39
Q4 (highest)   2.67    5.81     40

--- return_12m ---
                mean  median  count
roce_quartile                      
Q1 (lowest)    -3.00   -4.02     40
Q2              5.83    1.31     39
Q3             11.20    2.66     39
Q4 (highest)    7.18    2.24     40



## 6. Test 3 — Combined: High F-Score AND High ROCE vs the rest
This is the actual "quality" thesis your Long-Term mode is built on — does combining both
signals show a stronger edge than either alone?


In [6]:
merged['high_quality'] = (merged['piotroski_f_score'] >= 7) & (merged['roce'] >= merged['roce'].quantile(0.75))

print("=== High Quality (F-Score>=7 AND ROCE top quartile) vs Rest ===\n")
for period in ['return_3m', 'return_6m', 'return_12m']:
    high_q = merged[merged['high_quality']][period].dropna()
    rest = merged[~merged['high_quality']][period].dropna()
    print(f"--- {period} ---")
    print(f"High Quality: mean={high_q.mean()*100:.2f}%, median={high_q.median()*100:.2f}%, n={len(high_q)}")
    print(f"Rest:         mean={rest.mean()*100:.2f}%, median={rest.median()*100:.2f}%, n={len(rest)}")
    print(f"Edge (mean difference): {(high_q.mean() - rest.mean())*100:+.2f} percentage points\n")


=== High Quality (F-Score>=7 AND ROCE top quartile) vs Rest ===

--- return_3m ---
High Quality: mean=12.77%, median=11.35%, n=30
Rest:         mean=12.90%, median=11.38%, n=164
Edge (mean difference): -0.12 percentage points

--- return_6m ---
High Quality: mean=4.76%, median=7.44%, n=30
Rest:         mean=1.50%, median=0.19%, n=164
Edge (mean difference): +3.26 percentage points

--- return_12m ---
High Quality: mean=10.46%, median=8.09%, n=30
Rest:         mean=5.19%, median=0.57%, n=164
Edge (mean difference): +5.28 percentage points



In [7]:
high_quality_stocks = merged[merged['high_quality']][['symbol_short', 'return_12m']].sort_values('return_12m', ascending=False)
print(high_quality_stocks.to_string(index=False))

symbol_short  return_12m
        IDEA    0.917900
  NATIONALUM    0.885313
  CUMMINSIND    0.634059
         MCX    0.579130
         BSE    0.377784
  GMRAIRPORT    0.324099
        OFSS    0.307548
       LUPIN    0.285552
    GLENMARK    0.263783
    HINDZINC    0.211829
  HEROMOTOCO    0.185242
  BAJAJ-AUTO    0.184720
      MARICO    0.172426
    TATACOMM    0.138080
     HDFCAMC    0.102295
   SOLARINDS    0.059545
  PIDILITIND    0.053540
   POWERGRID   -0.008810
  PREMIERENE   -0.012018
  BHARTIARTL   -0.041416
   BRITANNIA   -0.068622
  INDUSTOWER   -0.089765
     HYUNDAI   -0.093253
     PAGEIND   -0.108634
  GODFRYPHLP   -0.266699
     HCLTECH   -0.312865
        INFY   -0.321864
         TCS   -0.371497
    JUBLFOOD   -0.386793
       TRENT   -0.461916


## 7. How to read all of this honestly

- Look at whether returns **increase step-by-step** from Low→Mid→High F-Score, and Q1→Q4 ROCE.
  A clean upward staircase is a good sign of real signal; a flat or messy pattern means no edge.
- The **combined High Quality test (Section 6)** is the most direct answer to "is your Long-Term
  mode thesis sound" — look for a meaningfully positive edge (ideally 5+ percentage points on
  6m/12m returns) that's consistent across both time periods.
- **Remember the look-ahead bias caveat from the top** — this uses CURRENT fundamentals against
  PAST price moves. If the result looks strong, that's encouraging but not final proof; a stock's
  fundamentals were likely similar 6-12 months ago for stable, high-quality companies (fundamentals
  don't flip overnight), so the bias is real but probably smaller here than for the technical
  signals in Step 3B (which are inherently point-in-time already, unlike this test).
- If NEITHER technical (Step 3B) NOR fundamental quality (this notebook) show real edge, that's an
  important signal to seriously reconsider scope — possibly focus the whole platform on
  explainability/research-tool value rather than prediction accuracy claims.


## 8. Next step depends on this result
- **If quality shows real edge:** rebuild Step 3's Long-Term and Position scoring around F-Score +
  ROCE as the primary drivers (already weighted heavily in your original design — this would
  validate that design choice), then move to Step 4 (Explainable AI + entry/target/stop-loss)
- **If quality also shows weak/no edge:** we need a bigger, more rigorous data engineering step —
  building true historical point-in-time fundamental snapshots (quarter by quarter, matched to
  forward returns from THAT quarter) — this removes the look-ahead bias caveat and gives a much
  more trustworthy answer, but it's a bigger build (a few more sessions of work)

Send me the 3 test outputs above and we'll decide the real next step from there.
